# TrustXAI: End-to-End Diagnostic Benchmark on Tabular Data

In this notebook, we demonstrate how to use `TrustXAI` to audit and compare two machine learning models:
1. **Random Forest Classifier**
2. **Logistic Regression (Calibrated Baseline)**

We evaluate:
- **Local Explanation Stability** ($L_E$ Lipschitz Continuity bound)
- **Conformal Prediction Sets** with 90% coverage guarantee ($\alpha = 0.10$)
- **Unified Trust Index (UTI)** and reliability classification (`CERTIFIED`, `FRAGILE`, `AMBIGUOUS_OR_OOD`)
- Automated export to academic $\LaTeX$ tables.


In [4]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"System path configured. Project root: {project_root}")

System path configured. Project root: D:\Development\projects for master\trustxai library


In [5]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load benchmark dataset
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

# 2. Three-way split: 60% Train, 20% Conformal Calibration, 20% Test Audit
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_cal, y_train, y_cal = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

# Standardize continuous features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_cal = scaler.transform(X_cal)
X_test = scaler.transform(X_test)

print("Dataset Split Summary:")
print(f" - Training set:    {X_train.shape[0]} samples")
print(f" - Calibration set: {X_cal.shape[0]} samples")
print(f" - Test set:        {X_test.shape[0]} samples")

Dataset Split Summary:
 - Training set:    341 samples
 - Calibration set: 114 samples
 - Test set:        114 samples


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# 1. Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

# 2. Train Logistic Regression
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)

# Baseline accuracy validation
print(f"Random Forest Test Accuracy:     {rf_model.score(X_test, y_test):.4f}")
print(f"Logistic Regression Test Accuracy: {lr_model.score(X_test, y_test):.4f}")

Random Forest Test Accuracy:     0.9474
Logistic Regression Test Accuracy: 0.9825


In [7]:
from trustxai.core.benchmark import TrustBenchmark
from trustxai.explainers.wrappers import ShapWrapper

# Extract predicted posterior probabilities
rf_probs_cal = rf_model.predict_proba(X_cal)
rf_probs_test = rf_model.predict_proba(X_test)

lr_probs_cal = lr_model.predict_proba(X_cal)
lr_probs_test = lr_model.predict_proba(X_test)

# --- 1. Audit Random Forest ---
print("Auditing Random Forest...")
rf_explainer = ShapWrapper(model=rf_model, explainer_type="tree", target_class=1)
rf_benchmark = TrustBenchmark(
    explainer=rf_explainer,
    alpha=0.10,               # Target coverage guarantee >= 90%
    epsilon=0.05,             # 5% relative perturbation radius
    num_perturbations=30,
    lipschitz_threshold=2.0
)
rf_benchmark.calibrate(X_train=X_train, probs_cal=rf_probs_cal, y_cal=y_cal)
rf_results = rf_benchmark.audit(X_test=X_test, probs_test=rf_probs_test)

# --- 2. Audit Logistic Regression ---
print("Auditing Logistic Regression...")
lr_explainer = ShapWrapper(
    model=lr_model, 
    background_data=X_train[:50], 
    explainer_type="kernel", 
    target_class=1
)
lr_benchmark = TrustBenchmark(
    explainer=lr_explainer,
    alpha=0.10,
    epsilon=0.05,
    num_perturbations=30,
    lipschitz_threshold=2.0
)
lr_benchmark.calibrate(X_train=X_train, probs_cal=lr_probs_cal, y_cal=y_cal)
lr_results = lr_benchmark.audit(X_test=X_test, probs_test=lr_probs_test)

print("Diagnostic audits completed successfully.")

Auditing Random Forest...
Auditing Logistic Regression...


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Diagnostic audits completed successfully.


In [8]:
def compile_metrics_dataframe(model_name: str, results) -> pd.DataFrame:
    records = []
    for idx, r in enumerate(results):
        records.append({
            "Sample_ID": idx,
            "Model": model_name,
            "UTI_Score": r.uti_score,
            "Status": r.status.value,
            "Lipschitz_Bound": r.lipschitz_bound,
            "Prediction_Set_Size": len(r.prediction_set),
            "OOD_Score": r.ood_score
        })
    return pd.DataFrame(records)

df_rf = compile_metrics_dataframe("Random Forest", rf_results)
df_lr = compile_metrics_dataframe("Logistic Regression", lr_results)
df_all = pd.concat([df_rf, df_lr], ignore_index=True)

# Generate cross-tabulation summary
summary_df = df_all.groupby(["Model", "Status"]).size().unstack(fill_value=0)
summary_df["Mean_UTI"] = df_all.groupby("Model")["UTI_Score"].mean().round(3)
summary_df["Mean_Lipschitz"] = df_all.groupby("Model")["Lipschitz_Bound"].mean().round(3)

summary_df

Status,AMBIGUOUS_OR_OOD,CERTIFIED,FRAGILE,Mean_UTI,Mean_Lipschitz
Model,,,,,
Logistic Regression,3,94,17,0.289,1.528
Random Forest,3,111,0,0.571,0.237


In [9]:
from trustxai.reporting.latex import AcademicReporter

benchmark_comparison = {
    "Random Forest": rf_results,
    "Logistic Regression": lr_results
}

latex_table_str = AcademicReporter.generate_summary_table(
    results_by_model=benchmark_comparison,
    caption="Empirical Comparison of Explanation Stability and Uncertainty Guarantees on Breast Cancer Benchmark",
    label="tab:diagnostic_breast_cancer"
)

# Render LaTeX code output
print(latex_table_str)

# Save artifact to file
output_path = Path("benchmark_results.tex")
output_path.write_text(latex_table_str, encoding="utf-8")

print(f"\nLaTeX table successfully exported to: {output_path.resolve()}")

\begin{table*}[t]
\centering
\caption{Empirical Comparison of Explanation Stability and Uncertainty Guarantees on Breast Cancer Benchmark}
\label{tab:diagnostic_breast_cancer}
\begin{tabular}{lcccc}
\toprule
\textbf{Model} & \textbf{Mean UTI} $\uparrow$ & \textbf{Certified (\%)} $\uparrow$ & \textbf{Fragile (\%)} $\downarrow$ & \textbf{Ambiguous/OOD (\%)} $\downarrow$ \\
\midrule
Random Forest & 0.571 & 97.4\% & 0.0\% & 2.6\% \\
Logistic Regression & 0.289 & 82.5\% & 14.9\% & 2.6\% \\
\bottomrule
\end{tabular}
\end{table*}

LaTeX table successfully exported to: D:\Development\projects for master\trustxai library\examples\benchmark_results.tex
